# Ollama Cloudflare Serving in Google Colab

This notebook allows you to run Ollama inside Google Colab (utilizing Colab's free/paid GPU resources) and securely expose it to the internet using a **Cloudflare Tunnel** (`cloudflared`). 

To make it easy for your backend services (such as the VibeSecurity backend) to dynamically find the server's URL, this script automatically updates a **GitHub Gist** with the latest Cloudflare tunnel URL whenever the notebook runs.

---

### Setup Instructions

To run this notebook, you need to configure two parameters in the configuration code block below:

1. **`GITHUB_TOKEN`**: A GitHub Personal Access Token (PAT) with the `gist` scope.
   - Go to your GitHub settings -> **Developer Settings** -> **Personal Access Tokens** -> **Tokens (classic)**.
   - Click **Generate new token (classic)**.
   - Give it a description (e.g., "Colab Ollama Tunnel") and check the **`gist`** scope.
   - Click **Generate token** and copy the token value.

2. **`GIST_ID`**: The unique identifier of a GitHub Gist.
   - Go to [gist.github.com](https://gist.github.com).
   - Create a new public or secret gist.
   - Name the file `ollama_url.txt` and put some dummy text (e.g., `http://localhost`) as its content.
   - Click **Create secret gist** or **Create public gist**.
   - Copy the Gist ID from the URL. The URL format is `https://gist.github.com/<username>/<gist_id>`. Copy the `<gist_id>` part (a 32-character hexadecimal string).

---

### How the Integration Works

1. **Colab Side or your preferred notebook (This Notebook)**:
   - Installs `zstd`, `ollama`, and `cloudflared`.
   - Starts the Ollama server in the background.
   - Launches a Cloudflare tunnel on port `11434`.
   - Monitors the tunnel logs to extract the generated `*.trycloudflare.com` URL.
   - Updates your Gist (`ollama_url.txt`) with the new URL using the GitHub API and your `GITHUB_TOKEN`.
   - Starts serving Ollama.

2. **Backend Side (`backend/core/llm_client.py`)**:
   - The backend reads the same `GIST_ID` (via configuration or env var).
   - On startup, it makes a GET request to the public GitHub Gist API to retrieve the contents of `ollama_url.txt`.
   - It updates its local `.env` file (`OLLAMA_URL_MAIN` / `OLLAMA_URL_HACKER`) with the fetched URL.
   - This allows the backend to communicate with the Colab-hosted Ollama models without requiring manual URL copying!


## Install Ollama inside this Colab

In [ ]:
!sudo apt-get install zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Download packages to create the Cloudflare tunnel

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb /
!dpkg -i cloudflared-linux-amd64.deb

### Prepare the thread to expose the Ollama via cloudflare

In [ ]:
import os
import subprocess
import threading
import time
import socket
import requests
import json

# Configurations
GITHUB_TOKEN = ""  # Replace with your GitHub Personal Access Token (PAT) with "gist" scope
GIST_ID = ""  # Replace with your GitHub Gist ID
FILENAME = "ollama_url.txt"
# -----------------------------------------------

# Set Ollama to run from any host (instead of this host only)
os.environ.update({'OLLAMA_HOST': '0.0.0.0'})

In [ ]:
def update_github_gist(url):
    """Updates the GitHub Gist with the new Cloudflare URL."""
    api_url = f"https://api.github.com/gists/{GIST_ID}"
    headers = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3+json"
    }
    data = {
        "files": {
            FILENAME: {"content": url}
        }
    }

    try:
        response = requests.patch(api_url, headers=headers, json=data)
        if response.status_code == 200:
            print(f"\nSuccessfully updated GitHub Gist with: {url}")
        else:
            print(f"\nFailed to update Gist. Status: {response.status_code}")
            print("Response:", response.text)
    except Exception as e:
        print(f"\nError updating Gist: {e}")

def start_cloudflare_tunnel(port):
    print("Waiting for Ollama server...")

    # Wait until the port is available
    while True:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.5):
                break
        except OSError:
            time.sleep(0.5)

    print("Ollama server detected. Starting Cloudflare Tunnel...")

    # Start Cloudflare tunnel
    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stderr=subprocess.PIPE,
        text=True
    )

    # Read tunnel output line by line to find the URL
    for line in process.stderr:
        if "trycloudflare.com" in line:
            # Extract the URL from the log line
            url = line[line.find("http"):].strip()
            url = "/".join(url.split("/")[:3])
            print(f"Tunnel URL found: {url}")

            # Update GitHub Gist
            update_github_gist(url)

# Run the tunnel function in a background thread
threading.Thread(
    target=start_cloudflare_tunnel,
    args=(11434,),
    daemon=True
).start()

### Start serving Ollama

In [ ]:
# Once the serve command is launched, the Colab waits indefinitely here for new command
!ollama serve